# E04 Breast Cancer：缩放、PCA 与召回权衡

> Issue #9 / E04

医疗场景不能只看 Accuracy：漏诊（假阴性）成本更高，需要看 Recall / PR。


In [ ]:
RANDOM_STATE=42
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from wanglaoshi import DataAnalyzer, CompetitionTools as CT
import pandas as pd, numpy as np

data = load_breast_cancer(as_frame=True)
df = data.frame
analyzer = DataAnalyzer(df)
display(analyzer.basic_statistics().head(10))

X = df.drop(columns=['target']); y = df['target']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=10, random_state=RANDOM_STATE)),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
pipe.fit(Xtr, ytr)
proba = pipe.predict_proba(Xte)[:,1]
pred = (proba >= 0.5).astype(int)
print(classification_report(yte, pred))
print('ROC-AUC', roc_auc_score(yte, proba))

# 阈值扫描：提高召回
for thr in [0.3, 0.4, 0.5, 0.6]:
    p = (proba >= thr).astype(int)
    from sklearn.metrics import recall_score, precision_score
    print(thr, 'precision', precision_score(yte,p), 'recall', recall_score(yte,p))

baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr, ytr, Xte, yte, task_type='classification')
baseline.compare_models(results, top_n=5)


## 检查清单
- [x] 分层划分
- [x] 缩放 + PCA 管线
- [x] 阈值-召回权衡
- [ ] 画 PR 曲线并选业务阈值
